In [74]:
import fitz
from PIL import Image
import io
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
import pytesseract

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

In [75]:
pdf_path = r"C:\Users\Admin\Downloads\labor-law.pdf"

In [76]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extracts text from a PDF file using OCR (Tesseract).
    """
    print("Extracting text from PDF using OCR...")
    text_raw = ""
    with fitz.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf):
            pix = page.get_pixmap(matrix=fitz.Matrix(4, 4), alpha=False)
            image = Image.open(io.BytesIO(pix.tobytes("png")))
            page_text = pytesseract.image_to_string(image, lang="ara", config="--psm 4")
            text_raw += f"\n\n--- Page {page_number + 1} ---\n{page_text}"
    return text_raw

In [77]:
def clean_arabic_text(raw_text: str) -> str:
    """
    Cleans the extracted Arabic text by removing noise, URLs, and extra spaces.
    """
    print("Cleaning the extracted text...")
    cleaned_data = []
    for line in raw_text.split('\n'):  
        line = line.strip()
        line = re.sub(r"\s+", " ", line)
        line = re.sub(r"http\S+|www\.\S+", "", line)
        line = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s،؛؟.,:()\-]", "", line)
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            cleaned_data.append(line)
    return "\n".join(cleaned_data)

In [78]:
def chunk_by_article(text: str) -> list[Document]:
    documents = []

    current_chapter = ""
    current_section = ""
    current_article_title = ""
    current_article_lines = []

    lines = text.splitlines()

    for raw_line in lines:
        line = raw_line.strip()

        if not line:
            continue

        if line.startswith("--- Page") or line.startswith("--- الصفحة"):
            continue

        if line.startswith("الباب"):
            current_chapter = line
            current_section = ""
            continue

        if line.startswith("الفصل"):
            current_section = line
            continue

        if line.startswith("المادة"):

            if current_article_title:
                article_text = "\n".join(
                    [current_article_title, *current_article_lines]
                ).strip()

                documents.append(
                    Document(
                        page_content=article_text,
                        metadata={
                            "chapter": current_chapter,
                            "section": current_section,
                            "article": current_article_title,
                            "source": "نظام العمل السعودي",
                        },
                    )
                )

            current_article_title = line
            current_article_lines = []
            continue

        if current_article_title:
            current_article_lines.append(line)


    if current_article_title:
        article_text = "\n".join(
            [current_article_title, *current_article_lines]
        ).strip()

        documents.append(
            Document(
                page_content=article_text,
                metadata={
                    "chapter": current_chapter,
                    "section": current_section,
                    "article": current_article_title,
                    "source": "نظام العمل السعودي",
                },
            )
        )

    return documents

In [79]:
def split_and_build_db(documents: list[Document],db_path: str):
    print("Applying secondary split...")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=3000,
        chunk_overlap=0,
        separators=["\n\n", "\n", " ", ""]
    )

    split_documents = text_splitter.split_documents(documents)

    final_documents = []

    for document in split_documents:
        article_title = document.metadata.get("article", "")

        content = document.page_content

        if article_title and not content.startswith(article_title):
            content = f"{article_title}\n{content}"

        final_documents.append(
            Document(
                page_content=content,
                metadata=document.metadata
            )
        )

    print(
        f"Prepared {len(final_documents)} final document chunks "
        f"from {len(documents)} legal articles."
    )

    embedding_model = HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-base"
    )

    Chroma.from_documents(
        documents=final_documents,
        embedding=embedding_model,
        persist_directory=db_path
    )

    print("Vector Database successfully built and saved!")

In [82]:
extact_data = extract_text_from_pdf(pdf_path)
cleaned_data = clean_arabic_text(extact_data)
chunks_data = chunk_by_article(cleaned_data)
len(chunks_data)

Extracting text from PDF using OCR...
Cleaning the extracted text...


243